## Assignment 4 — Cognitive FAQ System (Pandas)

**Roll No:** 1024170446  
**Name:** Paryag Bansal

## Q1 — Personalized knowledge base

In [1]:
import pandas as pd

fixed_entries = [
    {"question": "what is the annual fee", "answer": "The annual fee is Rs 500.",
     "keywords": "fee cost price charge", "category": "billing"},
    {"question": "how to reset password", "answer": "Go to Settings > Reset Password.",
     "keywords": "password reset login", "category": "account"},
    {"question": "what are your working hours", "answer": "We are open 9 AM to 5 PM.",
     "keywords": "hours timing open time", "category": "general"},
    {"question": "how can i pay the fee", "answer": "You can pay via UPI, card, or net banking.",
     "keywords": "pay payment upi fee", "category": "billing"},
]

# last two digits: 2 -> general, 3 -> billing
personal = [
    {
        "question": "how do i update my profile photo",
        "answer": "Open Profile > Edit Photo and upload a new image.",
        "keywords": "profile photo update picture",
        "category": "general"
    },
    {
        "question": "why was i charged twice for the fee",
        "answer": "Duplicate charges are reversed within 3-5 working days. Contact support with your transaction id.",
        "keywords": "charge double fee refund transaction",
        "category": "billing"
    },
]

df = pd.DataFrame(fixed_entries + personal)
df

,question,answer,keywords,category
0,what is the annual fee,The annual fee is Rs 500.,fee cost price charge,billing
1,how to reset password,Go to Settings > Reset Password.,password reset login,account
2,what are your working hours,We are open 9 AM to 5 PM.,hours timing open time,general
3,how can i pay the fee,"You can pay via UPI, card, or net banking.",pay payment upi fee,billing
4,how do i update my profile photo,Open Profile > Edit Photo and upload a new image.,profile photo update picture,general
5,why was i charged twice for the fee,Duplicate charges are reversed within 3-5 work...,charge double fee refund transaction,billing


## Q2 — Score a query

In [2]:
def score_query(query, df):
    q_words = set(query.lower().split())
    scores = []
    for i, row in df.iterrows():
        kw = set(row["keywords"].lower().split())
        conf = len(q_words & kw)
        if conf > 0:
            scores.append((conf, i, row["question"], row["answer"], row["category"]))
    scores.sort(reverse=True)
    return scores

# demo
results = score_query("how to pay the fee", df)
for conf, idx, q, a, cat in results:
    print(f"[{conf}] {q}  ({cat})")
    print(f"    -> {a}\n")

[2] how can i pay the fee  (billing)
    -> You can pay via UPI, card, or net banking.

[1] why was i charged twice for the fee  (billing)
    -> Duplicate charges are reversed within 3-5 working days. Contact support with your transaction id.

[1] what is the annual fee  (billing)
    -> The annual fee is Rs 500.



## Q3 — same_category()

In [ ]:
def same_category(category_name, df):
    return df[df["category"] == category_name]["question"].tolist()

# one of the personal categories was "billing"
print(same_category("billing", df))

## Q4 — Add keyword + save CSV

In [3]:
# pick first entry, add a new keyword
# in a real run you can use: new_kw = input("Enter a new keyword: ")
new_kw = "yearly"
df.at[0, "keywords"] = df.at[0, "keywords"] + " " + new_kw
print("Updated keywords for row 0:", df.at[0, "keywords"])

df.to_csv("1024170446_faq_data.csv", index=False)
print("Saved to 1024170446_faq_data.csv")
df.head()

Updated keywords for row 0: fee cost price charge yearly
Saved to 1024170446_faq_data.csv


,question,answer,keywords,category
0,what is the annual fee,The annual fee is Rs 500.,fee cost price charge yearly,billing
1,how to reset password,Go to Settings > Reset Password.,password reset login,account
2,what are your working hours,We are open 9 AM to 5 PM.,hours timing open time,general
3,how can i pay the fee,"You can pay via UPI, card, or net banking.",pay payment upi fee,billing
4,how do i update my profile photo,Open Profile > Edit Photo and upload a new image.,profile photo update picture,general


## Q5 — Count per category

In [4]:
print(df.groupby("category").size())

category
account    1
billing    3
general    2
dtype: int64


## Q6 — Handle ties in scoring

In [6]:
def score_query_ties(query, df):
    q_words = set(query.lower().split())
    scores = []
    for i, row in df.iterrows():
        kw = set(row["keywords"].lower().split())
        conf = len(q_words & kw)
        if conf > 0:
            scores.append((conf, i, row["question"], row["answer"], row["category"]))
    if not scores:
        print("No match found.")
        return
    scores.sort(reverse=True)
    top = scores[0][0]
    ties = [s for s in scores if s[0] == top]
    if len(ties) > 1:
        print(f"Multiple matches with score {top}:")
    for conf, idx, q, a, cat in ties:
        print(f"[{conf}] {q}  ({cat})")
        print(f"    -> {a}")
    return ties

print("Query that ties (fee):")
score_query_ties("fee charge", df)

print("\nQuery with a clear winner: ")
score_query_ties("reset password login", df)

Query that ties (fee):
Multiple matches with score 2:
[2] why was i charged twice for the fee  (billing)
    -> Duplicate charges are reversed within 3-5 working days. Contact support with your transaction id.
[2] what is the annual fee  (billing)
    -> The annual fee is Rs 500.

Query with a clear winner: 
[3] how to reset password  (account)
    -> Go to Settings > Reset Password.


[(3,
  1,
  'how to reset password',
  'Go to Settings > Reset Password.',
  'account')]